In [2]:
from google.colab import userdata
import os
os.environ["LANGCHAIN_TRACING"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = userdata.get('langsmith')
os.environ["LANGCHAIN_PROJECT"] = "Short term memory"

In [3]:
!pip install -U langchain-google-genai -q
from langchain_google_genai import ChatGoogleGenerativeAI

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.7 MB/s eta 0:00:00


In [5]:
API_key = userdata.get('gemini')

llm_model = "gemini-2.5-flash"
os.environ["GOOGLE_API_KEY"] = API_key
model = ChatGoogleGenerativeAI(
    model=llm_model,
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)


# No memory

In [6]:
from langchain.agents import create_agent


agent = create_agent(
    model = model
)

In [7]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Hello my name is Seán and my favourite colour is green")

response = agent.invoke(
    {"messages": [question]}
)

In [8]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Hello my name is Seán and my favourite colour is green', additional_kwargs={}, response_metadata={}, id='b3aaacb7-b015-4379-bfe4-9039b15a6e7b'),
              AIMessage(content='Hello Seán! Green is a wonderful choice – a very refreshing and vibrant color.\n\nNice to meet you!', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cbde0-0f6a-7c81-bca2-5a31a59ee9d3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 171, 'total_tokens': 184, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 148}})]}


In [9]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content="What's my favourite colour?", additional_kwargs={}, response_metadata={}, id='78427f2a-744e-4287-b490-d0c30d8bba39'),
              AIMessage(content="As an AI, I don't have access to your personal preferences or information, so I can't possibly know your favorite color!\n\nYou'll have to tell me! What is it?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cbde0-5a0e-7150-bfa6-b12e8e070749-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 536, 'total_tokens': 544, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 495}})]}


# with Memory

In [12]:
from langgraph.checkpoint.memory import InMemorySaver

In [14]:
agent = create_agent(
    model = model,
    checkpointer = InMemorySaver()
)

In [15]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Hello my name is Seán and my favourite colour is green")
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [question]},
    config,
)

In [16]:
pprint(response)

{'messages': [HumanMessage(content='Hello my name is Seán and my favourite colour is green', additional_kwargs={}, response_metadata={}, id='0af0d6b8-2918-4bff-924e-df48c09d7e72'),
              AIMessage(content='Hello Seán! Green is a wonderful choice – a very refreshing and vibrant color.\n\nNice to meet you!', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cbdfa-96fa-74c0-84d7-3b538b6911c4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 171, 'total_tokens': 184, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 148}})]}


In [17]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]},
    config,
)

pprint(response)

{'messages': [HumanMessage(content='Hello my name is Seán and my favourite colour is green', additional_kwargs={}, response_metadata={}, id='0af0d6b8-2918-4bff-924e-df48c09d7e72'),
              AIMessage(content='Hello Seán! Green is a wonderful choice – a very refreshing and vibrant color.\n\nNice to meet you!', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cbdfa-96fa-74c0-84d7-3b538b6911c4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 171, 'total_tokens': 184, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 148}}),
              HumanMessage(content="What's my favourite colour?", additional_kwargs={}, response_metadata={}, id='b1aa5d51-a5be-4002-9e2b-0bc9d6a1b100'),
              AIMessage(content='Your favourite colour is **green**!', additional_kwargs={}, response_metadata={'fi

In [18]:
response['messages'][-1].content

'Your favourite colour is **green**!'